# 📓 Notebook 1 — Data Exploration, Preprocessing & Normalization
**Project:** Indonesian Government Law Sentiment Analysis  
**Dataset:** EmoTweetID-Human.csv (2,243 labeled tweets)  
**Goal:** Clean and normalize the dataset, ready for fine-tuning XLM-RoBERTa

---
**Pipeline:**
```
Load CSV → Explore → Normalize text → Clean → Visualize → Save clean CSV
```

## ⚙️ Step 1 — Install & Import

In [ ]:
!pip install pandas numpy matplotlib seaborn emoji wordcloud scikit-learn -q
print('✅ Packages ready')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import re
import emoji
import json
from collections import Counter
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split

# Display settings
pd.set_option('display.max_colwidth', 120)
sns.set_theme(style='whitegrid', palette='muted')
print('✅ Imports done')

## 📂 Step 2 — Load Dataset

In [ ]:
# ============================================================
# Upload your EmoTweetID-Human.csv via the Colab file panel
# or mount Google Drive and set the path below
# ============================================================
CSV_PATH = 'EmoTweetID-Human.csv'   # ← change if needed

df = pd.read_csv(CSV_PATH)

# Drop unnamed index column if present
df = df.drop(columns=[c for c in df.columns if 'Unnamed' in c])

# Rename for consistency
df.columns = ['tweet', 'label']

print(f'✅ Loaded {len(df):,} rows')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 🔍 Step 3 — Data Exploration

In [ ]:
# --- Basic stats ---
print('=== DATASET OVERVIEW ===')
print(f'Total tweets     : {len(df):,}')
print(f'Null tweets      : {df["tweet"].isnull().sum()}')
print(f'Duplicate tweets : {df.duplicated(subset="tweet").sum()}')
print()

print('=== LABEL DISTRIBUTION ===')
label_counts = df['label'].value_counts()
label_pct    = df['label'].value_counts(normalize=True).mul(100).round(1)
label_summary = pd.DataFrame({'count': label_counts, 'percent': label_pct})
print(label_summary)
print()

print('=== TEXT LENGTH ===')
df['text_len'] = df['tweet'].str.len()
print(df['text_len'].describe().round(1))

In [ ]:
# --- Visualize label distribution ---
EMOTION_COLORS = {
    'anger':    '#e74c3c',
    'joy':      '#f1c40f',
    'fear':     '#9b59b6',
    'disgust':  '#27ae60',
    'sadness':  '#3498db',
    'surprise': '#e67e22'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
labels = label_counts.index.tolist()
colors = [EMOTION_COLORS.get(l, '#95a5a6') for l in labels]
bars = axes[0].bar(labels, label_counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Tweet Count per Emotion', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, label_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(val), ha='center', va='bottom', fontsize=11)

# Pie chart
axes[1].pie(label_counts.values, labels=labels, colors=colors,
            autopct='%1.1f%%', startangle=140,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Emotion Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('emotion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved')

In [ ]:
# --- Text length distribution per emotion ---
fig, ax = plt.subplots(figsize=(12, 5))
for label in df['label'].unique():
    subset = df[df['label'] == label]['text_len']
    subset.plot(kind='kde', ax=ax, label=label,
                color=EMOTION_COLORS.get(label), linewidth=2)

ax.set_title('Tweet Length Distribution per Emotion', fontsize=14, fontweight='bold')
ax.set_xlabel('Character Count')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig('length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧹 Step 4 — Build Normalization Dictionary (Kamus Alay)

This maps informal/slang Indonesian → formal Bahasa Indonesia.  
Feel free to **add more entries** you notice in your scraped data.

In [ ]:
# ============================================================
# KAMUS ALAY — Informal → Formal Bahasa Indonesia
# Add your own entries at the bottom of the dict
# ============================================================
KAMUS_ALAY = {
    # Pronouns
    'gw': 'saya', 'gue': 'saya', 'w': 'saya', 'aku': 'saya',
    'lo': 'kamu', 'lu': 'kamu', 'elo': 'kamu', 'loe': 'kamu',
    'dia': 'dia', 'dy': 'dia', 'dya': 'dia',
    'kita': 'kita', 'kami': 'kami',
    # Negation (critical for sentiment!)
    'gak': 'tidak', 'ga': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak',
    'enggak': 'tidak', 'kagak': 'tidak', 'tak': 'tidak',
    'ora': 'tidak',    # Javanese
    'henteu': 'tidak', # Sundanese
    'tdk': 'tidak', 'tdk': 'tidak', 'blm': 'belum', 'blum': 'belum',
    'bkn': 'bukan', 'bukan': 'bukan',
    # Common abbreviations
    'yg': 'yang', 'yng': 'yang',
    'dgn': 'dengan', 'dg': 'dengan',
    'krn': 'karena', 'karna': 'karena', 'kren': 'karena',
    'utk': 'untuk', 'tuk': 'untuk',
    'bgt': 'banget', 'banget': 'banget',
    'emg': 'memang', 'emang': 'memang',
    'jg': 'juga', 'juga': 'juga',
    'sdh': 'sudah', 'udah': 'sudah', 'udh': 'sudah', 'dah': 'sudah',
    'msh': 'masih', 'msih': 'masih',
    'sm': 'sama', 'ama': 'sama',
    'pd': 'pada', 'kpd': 'kepada',
    'dr': 'dari', 'dri': 'dari',
    'spy': 'supaya', 'biar': 'supaya',
    'tp': 'tapi', 'tpi': 'tapi',
    'klo': 'kalau', 'kalo': 'kalau', 'klw': 'kalau',
    'knp': 'kenapa', 'knapa': 'kenapa',
    'gmn': 'bagaimana', 'gimana': 'bagaimana',
    'bisa': 'bisa', 'bs': 'bisa',
    'lbh': 'lebih', 'lebih': 'lebih',
    'org': 'orang', 'orng': 'orang',
    'jd': 'jadi', 'jadinya': 'jadi',
    'punya': 'punya', 'pny': 'punya',
    'hrs': 'harus',
    'bnyk': 'banyak', 'byk': 'banyak',
    'spt': 'seperti', 'sprti': 'seperti', 'kyk': 'seperti', 'kayak': 'seperti',
    'krg': 'kurang',
    'trs': 'terus', 'trus': 'terus',
    'dgr': 'dengar',
    'mau': 'mau', 'mo': 'mau',
    'tau': 'tahu', 'tw': 'tahu',
    'ntar': 'nanti', 'nanti': 'nanti',
    'abis': 'habis', 'abiss': 'habis',
    'sampe': 'sampai', 'ampe': 'sampai',
    'bantu': 'bantu', 'bntu': 'bantu',
    'dpt': 'dapat', 'dpat': 'dapat',
    'hrs': 'harus',
    'lgsg': 'langsung',
    'bbrp': 'beberapa',
    'ttg': 'tentang',
    'thd': 'terhadap',
    'tsb': 'tersebut',
    # Javanese common words
    'iki': 'ini',
    'kuwi': 'itu',
    'ono': 'ada',
    'wis': 'sudah',
    'yo': 'ya',
    # Betawi / Jakarta slang
    'ane': 'saya',
    'ente': 'kamu',
    'nyokap': 'ibu',
    'bokap': 'ayah',
    # Twitter/internet slang
    'wkwk': 'haha',
    'wkwkwk': 'haha',
    'hehe': 'haha',
    'xixi': 'haha',
    'lol': 'lucu',
    'omg': 'astaga',
    'wtf': 'astaga',
    'btw': 'ngomong-ngomong',
    'oot': 'diluar topik',
    'otw': 'dalam perjalanan',
    'cmiiw': 'koreksi jika salah',
    'imho': 'menurut saya',
    'fr': 'sungguh',
    'ngl': 'jujur',
    'istg': 'serius',
    'literally': 'sungguh',
    'auto': 'otomatis',
    'gaskeun': 'lakukan saja',
    'santuy': 'santai',
    'kepo': 'ingin tahu',
    'baper': 'terbawa perasaan',
    'gabut': 'tidak ada kerjaan',
    # ============================================================
    # ADD YOUR OWN ENTRIES BELOW as you spot patterns in your data
    # ============================================================
}

print(f'✅ Kamus Alay loaded: {len(KAMUS_ALAY)} entries')

## 🧼 Step 5 — Text Cleaning & Normalization Functions

In [ ]:
def convert_emojis(text):
    """Convert emojis to text descriptions (sentiment carriers — don't remove)."""
    return emoji.demojize(text, language='en', delimiters=(' ', ' '))

def normalize_slang(text, kamus):
    """Replace informal/slang words with formal equivalents."""
    words = text.split()
    return ' '.join([kamus.get(w.lower(), w) for w in words])

def clean_tweet(text):
    """Full cleaning pipeline for a single tweet."""
    if not isinstance(text, str):
        return ''

    # 1. Lowercase
    text = text.lower()

    # 2. Convert emojis to text BEFORE removing symbols
    text = convert_emojis(text)

    # 3. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 4. Remove @mentions (keep hashtag text, remove # symbol)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)  # keep hashtag words, remove symbol

    # 5. Remove RT prefix
    text = re.sub(r'^rt\s+', '', text)

    # 6. Normalize repeated characters ("bangeeettt" → "banget")
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)  # max 2 repeated chars

    # 7. Normalize slang
    text = normalize_slang(text, KAMUS_ALAY)

    # 8. Remove special characters but keep apostrophes & hyphens in words
    text = re.sub(r'[^\w\s\'-]', ' ', text)

    # 9. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# --- Test on sample tweets ---
test_tweets = [
    "gak setuju bgt sama UU iki, literally ora masuk akal fr 😭",
    "wkwk auto marah gue liat kebijakan begini 🤣🔥",
    "emg pemerintah gak bisa diandalkan krn mereka ga pernah denger rakyat",
    "Semoga UU ini bisa lindungi anak2 kita dari bahaya medsos 🙏",
]

print('=== NORMALIZATION PREVIEW ===')
for t in test_tweets:
    print(f'BEFORE : {t}')
    print(f'AFTER  : {clean_tweet(t)}')
    print()

## ⚡ Step 6 — Apply Cleaning to Full Dataset

In [ ]:
print('🔄 Cleaning tweets...')
df['tweet_clean'] = df['tweet'].apply(clean_tweet)

# Remove any empty tweets after cleaning
before = len(df)
df = df[df['tweet_clean'].str.strip().str.len() > 3].reset_index(drop=True)
after = len(df)

print(f'✅ Done! {before - after} tweets removed (too short after cleaning)')
print(f'📊 Remaining: {after:,} tweets')
print()

# Compare before/after lengths
df['clean_len'] = df['tweet_clean'].str.len()
print('Text length comparison:')
print(pd.DataFrame({
    'original': df['text_len'].describe(),
    'cleaned':  df['clean_len'].describe()
}).round(1))

In [ ]:
# --- Side-by-side sample comparison ---
print('=== BEFORE vs AFTER SAMPLE ===')
sample = df[['tweet', 'tweet_clean', 'label']].sample(8, random_state=42)
for _, row in sample.iterrows():
    print(f'[{row["label"].upper()}]')
    print(f'  ORIGINAL : {row["tweet"]}')
    print(f'  CLEANED  : {row["tweet_clean"]}')
    print()

## 📊 Step 7 — Class Balance Check & Label Encoding

In [ ]:
# --- Encode labels to integers ---
LABEL2ID = {
    'anger':    0,
    'disgust':  1,
    'fear':     2,
    'joy':      3,
    'sadness':  4,
    'surprise': 5
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

df['label_id'] = df['label'].map(LABEL2ID)

print('Label encoding:')
print(LABEL2ID)
print()

# --- Compute class weights for weighted loss during training ---
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array(sorted(LABEL2ID.values())),
    y=df['label_id'].values
)

print('Class weights (for weighted loss in training):')
for label, weight in zip(LABEL2ID.keys(), class_weights):
    print(f'  {label:10s} → {weight:.4f}')

## ✂️ Step 8 — Stratified Train / Validation / Test Split

In [ ]:
# Stratified split: 80% train / 10% val / 10% test
train_df, temp_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print(f'Train : {len(train_df):,} tweets ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df):,}  tweets ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df):,}  tweets ({len(test_df)/len(df)*100:.1f}%)')
print()

# Verify stratification
print('Label distribution check (should be ~similar across splits):')
split_check = pd.DataFrame({
    'train': train_df['label'].value_counts(normalize=True).mul(100).round(1),
    'val':   val_df['label'].value_counts(normalize=True).mul(100).round(1),
    'test':  test_df['label'].value_counts(normalize=True).mul(100).round(1),
})
print(split_check)

## ☁️ Step 9 — Word Clouds per Emotion

In [ ]:
# Indonesian stopwords
STOPWORDS_ID = {
    'yang', 'dan', 'di', 'ke', 'dari', 'ini', 'itu', 'dengan', 'untuk',
    'pada', 'adalah', 'atau', 'juga', 'dalam', 'tidak', 'ada', 'kita',
    'saya', 'kamu', 'dia', 'mereka', 'kami', 'akan', 'sudah', 'belum',
    'bisa', 'harus', 'lebih', 'sangat', 'tapi', 'karena', 'kalau', 'sama',
    'jadi', 'lagi', 'masih', 'seperti', 'satu', 'dua', 'ya', 'aja', 'sih',
    'nih', 'deh', 'dong', 'loh', 'saja', 'pun', 'itu', 'pun', 'juga',
    'banget', 'mau', 'nanti', 'buat', 'tahu', 'mungkin', 'memang', 'terus',
    'udah', 'gitu', 'kayak', 'gak', 'ga', 'orang', 'sini', 'sana', 'situ',
    'the', 'a', 'an', 'is', 'in', 'on', 'at', 'to', 'of', 'and', 'for',
}

emotions = list(LABEL2ID.keys())
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, emotion in enumerate(emotions):
    text = ' '.join(df[df['label'] == emotion]['tweet_clean'].values)
    wc = WordCloud(
        width=600, height=400,
        background_color='white',
        stopwords=STOPWORDS_ID,
        colormap='RdYlGn' if emotion in ['joy'] else 'OrRd',
        max_words=80
    ).generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(f'{emotion.upper()}  ({label_counts[emotion]} tweets)',
                       fontsize=13, fontweight='bold',
                       color=EMOTION_COLORS.get(emotion, 'black'))
    axes[i].axis('off')

plt.suptitle('Word Clouds per Emotion Class', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('wordclouds_emotion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Word clouds saved')

## 💾 Step 10 — Save All Outputs

In [ ]:
# --- Save cleaned full dataset ---
df.to_csv('emotweet_cleaned.csv', index=False, encoding='utf-8-sig')
print('✅ Saved: emotweet_cleaned.csv')

# --- Save train/val/test splits ---
train_df.to_csv('split_train.csv', index=False, encoding='utf-8-sig')
val_df.to_csv('split_val.csv',   index=False, encoding='utf-8-sig')
test_df.to_csv('split_test.csv', index=False, encoding='utf-8-sig')
print('✅ Saved: split_train.csv, split_val.csv, split_test.csv')

# --- Save class weights as JSON (used in Notebook 2) ---
weights_dict = {str(i): float(w) for i, w in enumerate(class_weights)}
with open('class_weights.json', 'w') as f:
    json.dump(weights_dict, f)
print('✅ Saved: class_weights.json')

# --- Save label mappings as JSON (used in Notebooks 2 & 3) ---
with open('label_mapping.json', 'w') as f:
    json.dump({'label2id': LABEL2ID, 'id2label': ID2LABEL}, f)
print('✅ Saved: label_mapping.json')

print()
print('=== FILES READY FOR NOTEBOOK 2 ===')
print('  📄 emotweet_cleaned.csv')
print('  📄 split_train.csv')
print('  📄 split_val.csv')
print('  📄 split_test.csv')
print('  📄 class_weights.json')
print('  📄 label_mapping.json')

In [ ]:
# --- Optional: Download all files ---
from google.colab import files

for fname in ['emotweet_cleaned.csv', 'split_train.csv', 'split_val.csv',
              'split_test.csv', 'class_weights.json', 'label_mapping.json',
              'emotion_distribution.png', 'wordclouds_emotion.png']:
    files.download(fname)
    print(f'📥 Downloading {fname}')

---
## ✅ Notebook 1 Complete!

| Output file | Used in |
|---|---|
| `emotweet_cleaned.csv` | Reference / EDA |
| `split_train.csv` | Notebook 2 — fine-tuning |
| `split_val.csv` | Notebook 2 — validation |
| `split_test.csv` | Notebook 2 — final evaluation |
| `class_weights.json` | Notebook 2 — weighted loss |
| `label_mapping.json` | Notebook 2 & 3 — label decode |

**➡️ Next: Notebook 2 — Fine-tuning XLM-RoBERTa**